In [9]:
import threading
import time
def cpu_bound_task(n):
    count = 0
    for num in range(2,n):
        is_prime = True
        for i in range(2, int(num**0.5) + 1):
            if num % i == 0:
                is_prime = False
                break
        if is_prime:
            count += 1
    return count
start = time.perf_counter()
result1 = cpu_bound_task(100000)
result2 = cpu_bound_task(100000)
single_time = time.perf_counter() - start
print(f"单线程: {single_time:.3f}s, 结果: {result1}, {result2}")
start = time.perf_counter()
t1 = threading.Thread(target=cpu_bound_task,args=(100000,))
t2 = threading.Thread(target=cpu_bound_task,args=(100000,))
t1.start();t2.start()
t1.join();t2.join()
multi_time = time.perf_counter() - start
print(f"多线程: {multi_time:.3f}s")
print(f"加速比: {single_time / multi_time:.2f}x")

单线程: 0.152s, 结果: 9592, 9592
多线程: 0.154s
加速比: 0.98x


In [11]:
import threading
import time
def io_bound_task(task_id):
    print(f"  [任务{task_id}] 开始 I/O 等待...")
    time.sleep(2)
    print(f"  [任务{task_id}] I/O 完成")
    return task_id
start = time.perf_counter()
io_bound_task(1)
io_bound_task(2)
io_bound_task(3)
single_time = time.perf_counter() - start
print(f"\n单线程总耗时: {single_time:.3f}s")
start = time.perf_counter()
thread = [threading.Thread(target=io_bound_task,args=(i,)) for i in range(1,4)]
for t in thread: t.start()
for t in thread: t.join()
multi_time = time.perf_counter() - start
print(f"多线程总耗时: {multi_time:.3f}s")
print(f"加速比: {single_time / multi_time:.2f}x")

  [任务1] 开始 I/O 等待...
  [任务1] I/O 完成
  [任务2] 开始 I/O 等待...
  [任务2] I/O 完成
  [任务3] 开始 I/O 等待...
  [任务3] I/O 完成

单线程总耗时: 6.003s
  [任务1] 开始 I/O 等待...
  [任务2] 开始 I/O 等待...
  [任务3] 开始 I/O 等待...
  [任务2] I/O 完成
  [任务3] I/O 完成
  [任务1] I/O 完成
多线程总耗时: 2.003s
加速比: 3.00x


In [ ]:
import multiprocessing
import time
def cpu_bound_task(n):
    count = 0
    for num in range(2,n):
        is_prime = True
        for i in range(2,int(num**0.5)+1):
            if num % i == 0:
                is_prime = False
                break
            if is_prime:
                count += 1
    return count
if __name__ == "__main__":
    start = time.perf_counter()
    with multiprocessing.Pool(processes=2) as pool:
        results = pool.map(cpu_bound_task,[100,100,100,100])
    multi_proc_time = time.perf_counter() - start
    print(f"多进程(4核): {multi_proc_time:.3f}s, 结果: {results}")

    start = time.perf_counter()
    for _ in range(4):
        cpu_bound_task(100)
    single_proc_time = time.perf_counter() - start
    print(f"单进程: {single_proc_time:.3f}s")
    print(f"加速比: {single_proc_time / multi_proc_time:.2f}x")

In [3]:
import threading
import time
import queue
import random
def producer(q,name,count):
    for i in range(count):
        item = f'{name}-item-{i}'
        q.put(item)
        print(f"  [生产] {name} 放入: {item}")
        time.sleep(random.uniform(0.1,0.3))
    print(f"  [生产] {name} 完成")
def consumer(q,name,stop_event):
    while not stop_event.is_set():
        try:
            item = q.get(timeout=0.5)
            print(f"  [消费] {name} 取出: {item}")
            time.sleep(random.uniform(0.2,0.5))
            q.task_done()
        except queue.Empty:
            continue
q = queue.Queue(maxsize=5)
stop_event = threading.Event()

producers = [
    threading.Thread(target=producer,args=(q,f"P{i}",5))
    for i in range(2)
]
consumers = [
    threading.Thread(target=consumer,args=(q,f"C{i}",stop_event))
    for i in range(3)
]
for t in producers + consumers:
    t.start()
for t in producers:
    t.join()
q.join()
print("\n所有任务处理完毕，通知消费者退出")
stop_event.set()
for t in consumers:
    t.join()
print("所有线程已退出")

  [生产] P0 放入: P0-item-0
  [生产] P1 放入: P1-item-0
  [消费] C0 取出: P0-item-0
  [消费] C1 取出: P1-item-0
  [生产] P0 放入: P0-item-1
  [消费] C2 取出: P0-item-1
  [生产] P1 放入: P1-item-1
  [生产] P0 放入: P0-item-2
  [生产] P1 放入: P1-item-2
  [生产] P0 放入: P0-item-3
  [消费] C0 取出: P1-item-1
  [消费] C1 取出: P0-item-2
  [生产] P0 放入: P0-item-4
  [生产] P1 放入: P1-item-3
  [消费] C2 取出: P1-item-2
  [消费] C1 取出: P0-item-3
  [生产] P0 完成
  [消费] C2 取出: P0-item-4
  [生产] P1 放入: P1-item-4
  [消费] C0 取出: P1-item-3
  [生产] P1 完成
  [消费] C1 取出: P1-item-4

所有任务处理完毕，通知消费者退出
所有线程已退出


In [1]:
import functools
import time
import logging
logging.basicConfig(level=logging.INFO,format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)
def logged(level=logging.INFO):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args,**kwargs):
            func_name = f"{func.__module__}.{func.__qualname__}"
            logger.log(level,f"调用{func_name}(args={args},kwargs={kwargs})")
            start = time.perf_counter()
            try:
                result = func(*args,**kwargs)
                elapsed = (time.perf_counter() - start) * 1000
                logger.log(level,f"{func_name} 返回 {result!r} (耗时 {elapsed:.2f}ms)")
                return result
            except Exception as e:
                elapsed = (time.perf_counter() - start) * 1000
                logger.log(logging.ERROR,f"{func_name} 抛出 {type(e).__name__}: {e} (耗时 {elapsed:.2f}ms)")
                raise
        return wrapper
    return decorator
@logged()
def divide(a,b):
    return a / b
@logged(level=logging.DEBUG)
def slow_add(a, b):
    time.sleep(0.5)
    return a + b
print(divide(10, 3))
try:
    divide(1, 0)
except ZeroDivisionError:
    pass

print(slow_add(1, 2))

2026-09-02 19:20:48,212 [INFO] 调用__main__.divide(args=(10, 3),kwargs={})
2026-09-02 19:20:48,213 [INFO] __main__.divide 返回 3.3333333333333335 (耗时 0.00ms)
2026-09-02 19:20:48,213 [INFO] 调用__main__.divide(args=(1, 0),kwargs={})
2026-09-02 19:20:48,214 [ERROR] __main__.divide 抛出 ZeroDivisionError: division by zero (耗时 0.00ms)


3.3333333333333335
3
